# ASAF tutorial #

**author:** Bartosz Mazur *(bartosz.mazur@pwr.edu.pl)*
***

In this tutorial you will learn how to analyze gas adsorption data calculated with transition matrix Monte Carlo method using *Adsorption Simulation Analysis Facilitator **(ASAF)*** python library. 

Note: This tutorial and the described library are still under development. This tutorial is for introductory purposes only. For complete information on the available functionalities, read the documentation. 

In [8]:
import numpy as np
from asaf.mpd import MPD

We will begin with example of water adsorption (using TIP4P water model) on MOF-303 at 298 K. All RASPA simulation input files needed to reproduce transition probability data you will find in folder named *raspa_input*. 

File `data/prob_MOF-303-E4D4_3.2.3_298.000000_1700.csv` contains transition probability data and is composed with columns:

- **macrostate**: with the number of molecules

- **P_up**: with the probability of inserting a molecule $P(N \rightarrow N+1)$

- **P_down**: with the probability of deleting a molecule $P(N \rightarrow N-1)$

- **term_1**: with the average potential energy at given macrostate used to extrapolate $\ln\Pi$ (as the first term of Taylor expansion in the grand canonical ensemble)

- **term_2**: with the fluctuations of the potential energy at given macrostate used to extrapolate $\ln\Pi$ (as the second term of Taylor expansion in the grand canonical ensemble). It is calculated using equation $<E^2> - <E>^2$, where $E$ is the potential energy at given macrostate

The first three columns are used directly to calculate the macrostate probability distribution (MPD), and the subsequent columns (term 1, 2, and following) are used for temperature extrapolation, as will be demonstrated later.  

To initialize the MPD from the probability file, we use the `prob_from_csv` function. We do not need anything else to calculate the MPD, but to calculate the adsorption isotherm, we need to know the conditions set in the simulation: temperature and fugacity. This information is obtained directly from the simulation metadata file. It must have the same name, except with a `.metadata.json` extension. 

In [10]:
mpd_mof303 = MPD.prob_from_csv(file_name="data/prob_MOF-303-E4D4_3.2.3_298.000000_1700.csv")
mpd_mof303.plot()

If you want to investigate in more detail how MPD changes with fugacity, you can use the `reweight_to_fug` function to do so. 

In [12]:
mpd_mof303.reweight_to_fug(500, inplace=True)
mpd_mof303.plot()

To calculate the adsorption isotherm, we use the function `calculate_isotherm`, to which we provide a list of pressures. Optionally, we can provide a saturation pressure to automatically calculate relative pressures and a fugacity coefficient to convert pressures to fugacities. By default, the fugacity coefficient is 1.0. 

This function can return `pandas.DataFrame` if `return_dataframe=True` or a new `Isotherm` object that has its own methods, so that further work on the isotherm can be done. 

In [14]:
saturation_pressure_298 = 4549.851434377435
fugacity_coefficient_298 = 0.9792420676107021
pressures = np.linspace(1, saturation_pressure_298, 50)

In [18]:
isotherm_mof303_298 = mpd_mof303.calculate_isotherm(
    pressures=pressures, 
    fugacity_coefficient=fugacity_coefficient_298,
    saturation_pressure=saturation_pressure_298,
    return_dataframe=False
)

We can print `Isotherm` object data with `dataframe` function.

In [19]:
isotherm_mof303_298.dataframe

,pressure,p/p0,fugacity,uptake
0,1.000000,0.000220,0.979242,4.014024
1,93.833703,0.020623,91.885909,5.471802
2,186.667405,0.041027,182.792576,6.661719
3,279.501108,0.061431,273.699243,7.873763
4,372.334811,0.081834,364.605910,9.630329
5,465.168514,0.102238,455.512577,14.305230
6,558.002216,0.122642,546.419244,24.725842
7,650.835919,0.143046,637.325911,28.628757
8,743.669622,0.163449,728.232578,30.182483
9,836.503325,0.183853,819.139245,30.987928


To plot isotherm we can use function `print`.

In [20]:
isotherm_mof303_298.plot()

We can easily save isotherm to a file with the functions `to_csv` and `to_aif`, which will write the isotherm to the corresponding file types. 

In [21]:
isotherm_mof303_298.to_csv('data/isotherm_mof303_298.csv')
isotherm_mof303_298.to_aif('data/isotherm_mof303_298.aif')

The great advantage of TMMC calculations is the information not only on equilibrium states, but also on metastable states. In the case of water adsorption in MOF-303 at 298 K we do not observe such, so below is also an analysis of water adsorption in NU-1000 at 298 K, where we observe long metastable phases in the gas and liquid branches. 

In [22]:
mpd_nu1000 = MPD.prob_from_csv('data/prob_NU-1000-noH2O_1.1.2_298.000000_3200.csv')
mpd_nu1000.plot()

In [23]:
isotherm_nu1000_298 = mpd_nu1000.calculate_isotherm(
    pressures=pressures, 
    fugacity_coefficient=fugacity_coefficient_298,
    saturation_pressure=saturation_pressure_298,
    return_dataframe=False
)

In [24]:
isotherm_nu1000_298.plot()